# RAG with ChromaDB for Predictive Diagnostics

## 1. Define the Augmentation Guide

The RAG system will transform a binary prediction ("maintenance needed: yes/no") into a rich diagnostic report that includes:

- Explanation of why the prediction was made (linking sensor readings to potential failures).
- Recommended actions (e.g., "Inspect oil pump", "Replace coolant hose").
- References to technical manuals or past work orders.
- Confidence level and urgency.

## 2. Prepare the Knowledge Base

Collect domain-specific documents that will serve as the retrieval corpus:

|Document Type | Examples	| Format |
|--------------|------------|--------|
|Maintenance manuals |	Engine repair guide, transmission service procedures	|PDF, HTML, text|
|Historical work orders	|Descriptions of past repairs with symptoms and actions	|CSV, JSON, text|
|Technical bulletins	|Manufacturer advisories on known issues |	PDF, text|
|Expert troubleshooting guides	|Step-by-step diagnostic flowcharts	| Text, markdown|

**Processing steps:**

- Extract text using tools like pypdf, tika, or textract.
- Clean and normalize text (remove headers/footers, fix encoding).
- Chunk documents into overlapping segments (500-1000 tokens) to preserve context.
- Assign metadata: source (filename), vehicle_model, component, date, etc.


In [ ]:
# Install if not exist
!pip install langchain-text-splitters

In [6]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_and_chunk_documents(directory):
    chunks = []
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            with open(os.path.join(directory, filename), 'r', encoding='utf-8') as f:
                text = f.read()
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=500,
                chunk_overlap=100,
                separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
            )
            doc_chunks = splitter.split_text(text)
            for i, chunk in enumerate(doc_chunks):
                chunks.append({
                    "text": chunk,
                    "metadata": {
                        "source": filename,
                        "chunk_id": i,
                        "vehicle_model": "truck" if "truck" in filename else "van"
                    }
                })
    return chunks

## 3. Set Up ChromaDB

```bash 
pip install chromadb sentence-transformers
```

In [ ]:
import chromadb
from chromadb.config import Settings

# Persistent client (data saved to ./chroma_data)
client = chromadb.PersistentClient(
    path="./chroma_data",
    settings=Settings(anonymized_telemetry=False)  # disable usage stats
)

## 4. Create a Collection and Choose an Embedding Function

In [ ]:
from sentence_transformers import SentenceTransformer

# Create embedding function using sentence-transformers
class SentenceTransformerEmbeddingFunction:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
    
    def __call__(self, texts):
        return self.model.encode(texts).tolist()

embedding_func = SentenceTransformerEmbeddingFunction()

# Create or get collection
collection_name = "maintenance_kb"
collection = client.get_or_create_collection(
    name=collection_name,
    embedding_function=embedding_func,
    metadata={"hnsw:space": "cosine"}  # use cosine similarity
)

## 5. Ingest Documents into ChromaDB

In [ ]:
def ingest_documents(chunks, batch_size=100):
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]
        ids = [f"doc_{i+j}" for j in range(len(batch))]
        documents = [chunk["text"] for chunk in batch]
        metadatas = [chunk["metadata"] for chunk in batch]
        
        collection.add(
            ids=ids,
            documents=documents,
            metadatas=metadatas
        )
        print(f"Added batch {i//batch_size + 1}")

# Example usage
chunks = load_and_chunk_documents("data/maintenance_texts")
ingest_documents(chunks)

## 6. Build the Retrieval Logic

In [ ]:
def retrieve_relevant_docs(query, n_results=5, filter_criteria=None):
    """
    Query ChromaDB collection.
    
    Args:
        query: Natural language query string.
        n_results: Number of documents to retrieve.
        filter_criteria: Optional metadata filter (e.g., {"component": "engine"}).
    
    Returns:
        List of dictionaries with 'document', 'metadata', 'distance'.
    """
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=filter_criteria  # metadata filter
    )
    
    # Format results
    retrieved = []
    for i in range(len(results['ids'][0])):
        retrieved.append({
            "text": results['documents'][0][i],
            "metadata": results['metadatas'][0][i],
            "distance": results['distances'][0][i] if 'distances' in results else None
        })
    return retrieved

## 7.Construct a Query from Vehicle Data and Prediction

In [ ]:
def build_query_from_telemetry(features, prediction, feature_importance):
    """
    features: dict of sensor readings
    prediction: 0 or 1
    feature_importance: dict mapping feature name to importance score
    """
    # Start with prediction statement
    query_parts = [f"Maintenance {'needed' if prediction else 'not needed'}."]
    
    # Add concerning features
    concerning = []
    if features.get('oil_pressure', 100) < 30:
        concerning.append(f"oil pressure low ({features['oil_pressure']:.1f} psi)")
    if features.get('coolant_temp', 0) > 105:
        concerning.append(f"coolant temperature high ({features['coolant_temp']:.1f}°C)")
    if features.get('mileage_since_last', 0) > 25000:
        concerning.append(f"high mileage since last maintenance ({features['mileage_since_last']:.0f} mi)")
    
    if concerning:
        query_parts.append("Vehicle has " + ", ".join(concerning) + ".")
    
    # Add top features by importance (if available)
    if feature_importance:
        top_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)[:3]
        query_parts.append("Key factors: " + ", ".join([f"{f.replace('_',' ')} ({v:.2f})" for f,v in top_features]))
    
    return " ".join(query_parts)

## 8. Generate Augmented Explanation with an LLM

In [ ]:
import ollama

def generate_diagnostic_report(query, retrieved_docs):
    context = "\n\n".join([f"[Source: {doc['metadata']['source']}]\n{doc['text']}" for doc in retrieved_docs])
    
    prompt = f"""
    You are an expert fleet maintenance advisor. Based on the vehicle data and technical documents below, provide a concise diagnostic report.
    
    VEHICLE DATA: {query}
    
    RELEVANT TECHNICAL DOCUMENTS:
    {context}
    
    Please structure your answer with:
    1. **Summary**: Brief overview of the situation.
    2. **Why maintenance may be needed**: Connect vehicle readings to potential issues.
    3. **Recommended actions**: Specific parts to inspect or replace, with priority.
    4. **References**: Mention the documents used.
    
    Be practical and base your answer only on the provided documents.
    """
    
    response = ollama.generate(model='mistral', prompt=prompt)
    return response['response']

## 9. Integrate with FastAPI

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI()

# Load XGBoost model
xgb_model = joblib.load("../models/predictive_maintenance_xgboost.pkl")
feature_names = xgb_model.get_booster().feature_names

# ChromaDB collection already initialized
# collection is global

class TelemetryRequest(BaseModel):
    vehicle_id: str
    mileage: float
    mileage_since_last: float
    engine_hours: float
    oil_pressure: float
    coolant_temp: float
    battery_voltage: float
    fuel_efficiency: float
    vehicle_age: float

class DiagnosticResponse(BaseModel):
    prediction: int
    confidence: float
    explanation: str
    sources: list
    recommended_actions: list

@app.post("/diagnostics/augmented", response_model=DiagnosticResponse)
async def augmented_diagnostics(request: TelemetryRequest):
    try:
        # Prepare features (exclude vehicle_id)
        features = request.dict(exclude={'vehicle_id'})
        feature_array = np.array([list(features.values())])
        
        # Prediction
        pred = int(xgb_model.predict(feature_array)[0])
        probs = xgb_model.predict_proba(feature_array)[0]
        confidence = float(probs[pred])
        
        # Feature importance (use model's global importance as proxy; for per-instance, use SHAP)
        importance = dict(zip(feature_names, xgb_model.feature_importances_))
        
        # Build query
        query = build_query_from_telemetry(features, pred, importance)
        
        # Retrieve relevant documents
        retrieved = retrieve_relevant_docs(query, n_results=3)
        
        # Generate explanation
        explanation = generate_diagnostic_report(query, retrieved)
        
        # Extract actions (simple heuristic)
        actions = [line.strip() for line in explanation.split('\n') 
                   if any(kw in line.lower() for kw in ['inspect', 'replace', 'check', 'schedule'])]
        
        return DiagnosticResponse(
            prediction=pred,
            confidence=confidence,
            explanation=explanation,
            sources=[{"text": r['text'][:200] + "...", "metadata": r['metadata']} for r in retrieved],
            recommended_actions=actions[:5]
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

## 10. Testing the Endpoint

Run the FastAPI and test with a sample request

```bash
uvicorn main:app --reload
```

In [ ]:
import requests

data = {
    "vehicle_id": "TRUCK-101",
    "mileage": 120000,
    "mileage_since_last": 28000,
    "engine_hours": 3000,
    "oil_pressure": 28,
    "coolant_temp": 102,
    "battery_voltage": 12.3,
    "fuel_efficiency": 8.5,
    "vehicle_age": 5
}

response = requests.post("http://localhost:8000/diagnostics/augmented", json=data)
print(response.json())

## 11. Production Deployment Considerations

- a. ChromaDB as a Separate Service

For production, run ChromdaDB in client/server mode to allow multiple services to connect and for better scalability.

In [ ]:
# Start ChromaDB server (separate process)
chroma run --path /path/to/persist --port 8000

# Then in your app, connect via HttpClient
import chromadb
client = chromadb.HttpClient(host='localhost', port=8000)

In [ ]:
# To add new documents
collection.add(
    ids=["new_doc_1"],
    documents=["New maintenance procedure text..."],
    metadatas=[{"source": "new_manual.txt"}]
)